# Rule Development

Write, test, and iterate on detection rules interactively.
Rules inspect the `AnalysisGraph` and yield `Finding` objects
for each vulnerability pattern they detect.

In [1]:
%load_ext autoreload
%autoreload 2
from confusion_sast.notebook import *

## Setup: Load a Known-Vulnerable Exercise

`r01/e03` (order overwrite) has multiple vulnerability patterns:
mixed JSON/form sources, dict merge overwrites, and conditional
source selection.

In [2]:
t = targets()
g = load(t.r01.e03)
g

Methods,Rule,Sources,Keys,Flags
POST,/orders,form,"delivery_address, items",
POST,/cart//items,json,item_id,
POST,/cart//checkout,"form, json",,MULTI-SOURCE
POST,/e2e/balance,json,"balance, user_id",


## Existing Rules

The rule registry tracks all built-in detection rules.

In [3]:
from confusion_sast.detection.rules import get_all_rules

rules = get_all_rules()
for rule_id, fn in rules.items():
    print(f"{rule_id}: {fn.__name__}")

CONF-001: dual_source_confusion
CONF-002: dual_parameter_confusion
CONF-003: cardinality_confusion
CONF-004: values_merge_confusion
CONF-005: dict_merge_overwrite
CONF-006: middleware_handler_divergence
CONF-007: conditional_source_selection


## Running Specific Rules

In [4]:
# Run a single rule by ID
results = run_rule("CONF-005", g)
print(f"CONF-005 found {len(results)} finding(s)")
for f in results:
    show(f)

CONF-005 found 1 finding(s)
[MEDIUM] CONF-005: Dict merge with user-controlled data: {**user_data, **safe_order_data}
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  In routes.checkout_cart, a dict merge unpacks user-controlled data alongside other values. User data is unpacked before safe data (safe values win on collision). Fields like 'total', 'user_id', or 'order_id' in the user data could overwrite computed values.
  Evidence:
    - {**user_data, **safe_order_data} at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:200


In [5]:
# Review those matches in the workbench
Explorer(g, results, target_path=t.r01.e03.path)

In [6]:
results = run_rule("CONF-007", g)
print(f"CONF-007 found {len(results)} finding(s)")
for f in results:
    show(f)

CONF-007 found 1 finding(s)
[MEDIUM] CONF-007: Conditional source selection with dict merge
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  Endpoint routes.checkout_cart reads from both request.json and request.form (likely via a conditional like `request.json if request.is_json else request.form`), then merges user-controlled data into a dict. The effective source depends on Content-Type, which is attacker-controlled.
  Evidence:
    - json at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181 in routes.checkout_cart
    - form at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181 in routes.checkout_cart


## Writing an Ad-Hoc Rule

The `@detect` decorator and `finding()` builder reduce boilerplate.
A rule function receives an `AnalysisGraph` and yields findings.

In [7]:
@detect("MY-001", severity="high")
def mixed_json_form(g):
    """Flag endpoints that read from both JSON and form sources."""
    for handler, route, accs in g.by_endpoint():
        json_accs = [a for a in accs if a.source == InputSource.JSON]
        form_accs = [a for a in accs if a.source == InputSource.FORM]
        if json_accs and form_accs:
            yield finding(
                f"Endpoint uses both JSON and form: {route.handler_name}",
                evidence=json_accs + form_accs,
                endpoint=route,
            )

results = run_fn(mixed_json_form, g)
show(results)

Found 1 potential confusion vulnerabilities:

[HIGH] MY-001: Endpoint uses both JSON and form: checkout_cart
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  Endpoint uses both JSON and form: checkout_cart
  Evidence:
    - json at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181 in routes.checkout_cart
    - form at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181 in routes.checkout_cart

Summary: {'high': 1}


## A More Complex Rule

This rule looks for endpoints where `request.form` is passed directly
(as a dict-like object) rather than accessed by key -- a pattern that
can smuggle unexpected fields.

In [8]:
@detect("MY-002", severity="medium")
def direct_source_passthrough(g):
    """Flag direct passthrough of request source objects."""
    for handler, route, accs in g.by_endpoint():
        direct = [a for a in accs if a.accessor == AccessorKind.DIRECT]
        if direct:
            yield finding(
                f"Direct source passthrough in {route.handler_name}: "
                + ", ".join(a.source.value for a in direct),
                evidence=direct,
                endpoint=route,
            )

results = run_fn(direct_source_passthrough, g)
show(results)

Found 4 potential confusion vulnerabilities:

[MEDIUM] MY-002: Direct source passthrough in create_new_order: form
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:92
  Endpoint: POST /orders (routes.create_new_order)
  Direct source passthrough in create_new_order: form
  Evidence:
    - form at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:92 in routes.create_new_order

[MEDIUM] MY-002: Direct source passthrough in add_item_to_cart_endpoint: json
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:154
  Endpoint: POST /cart/<cart_id>/items (routes.add_item_to_cart_endpoint)
  Direct source passthrough in add_item_to_cart_endpoint: json
  Evidence:
    - json at /Users/irina/src/unsafe-code/vulnerabilities/python/fla

## Batch Rule Testing

Instead of manually looping over exercises, use `batch_load()` to grab
all graphs in a section and `batch_run()` to execute a rule against them.
The resulting `BatchResult` renders as a summary table in Jupyter and
offers drill-down helpers.

In [9]:
graphs = batch_load(t.r01)
graphs

{'r01/e00': AnalysisGraph(7 routes, 10 accesses, 48 nodes, 73 edges),
 'r01/e01': AnalysisGraph(7 routes, 12 accesses, 51 nodes, 77 edges),
 'r01/e02': AnalysisGraph(7 routes, 13 accesses, 55 nodes, 82 edges),
 'r01/e03': AnalysisGraph(10 routes, 15 accesses, 68 nodes, 117 edges),
 'r01/e04': AnalysisGraph(9 routes, 19 accesses, 63 nodes, 105 edges),
 'r01/e05': AnalysisGraph(10 routes, 21 accesses, 71 nodes, 121 edges),
 'r01/e06': AnalysisGraph(12 routes, 28 accesses, 92 nodes, 156 edges),
 'r01/e07': AnalysisGraph(12 routes, 26 accesses, 103 nodes, 174 edges),
 'r01/e08': AnalysisGraph(12 routes, 26 accesses, 102 nodes, 173 edges)}

In [10]:
# Run the ad-hoc rule across all r01 exercises
results = batch_run(mixed_json_form, graphs)
results

Exercise,Findings,Rules,Severity
r01/e00,0,--,--
r01/e01,0,--,--
r01/e02,0,--,--
r01/e03,1,MY-001,HIGH
r01/e04,1,MY-001,HIGH
r01/e05,1,MY-001,HIGH
r01/e06,2,MY-001,HIGH
r01/e07,1,MY-001,HIGH
r01/e08,1,MY-001,HIGH


In [11]:
# Which exercises actually triggered findings?
results.hits

{'r01/e03': [Finding(rule_id='MY-001',
                     title='Endpoint uses both JSON and form: checkout_cart',
                     description='Endpoint uses both JSON and form: checkout_cart',
                     severity=<Severity.HIGH: 'high'>,
                     location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py',
                                       line=181,
                                       col=16),
                     evidence=[InputAccessFact(function_qualname='routes.checkout_cart',
                                               location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py',
                                                                 line=181,
                                                                 col=16),
                         

In [12]:
# Drill into a specific exercise's findings
for name in results.hits:
    print(f"--- {name} ---")
    for f in results[name]:
        show(f)

--- r01/e03 ---
[HIGH] MY-001: Endpoint uses both JSON and form: checkout_cart
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  Endpoint uses both JSON and form: checkout_cart
  Evidence:
    - json at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181 in routes.checkout_cart
    - form at /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py:181 in routes.checkout_cart
--- r01/e04 ---
[HIGH] MY-001: Endpoint uses both JSON and form: checkout_cart
  Location: /Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e04_negative_tip/routes.py:140
  Endpoint: POST /cart/<cart_id>/checkout (routes.checkout_cart)
  Endpoin

## Progressive Rule Refinement

The batch workflow makes it easy to iterate on a rule: write a first
draft, test it across exercises, tighten the logic, and verify the
change in detection coverage.

In [13]:
# Draft: flag ANY endpoint that accesses more than one input source
@detect("DRAFT-001", severity="medium")
def multi_source_naive(g):
    """Flag endpoints reading from multiple input sources (broad)."""
    for handler, route, accs in g.by_endpoint():
        sources = {a.source for a in accs}
        if len(sources) > 1:
            yield finding(
                f"Multiple sources in {route.handler_name}: {sorted(s.value for s in sources)}",
                evidence=accs,
                endpoint=route,
            )

draft_results = batch_run(multi_source_naive, graphs)
print(f"Draft rule: {draft_results.total} findings across {len(draft_results.hits)} exercises")
draft_results

Draft rule: 8 findings across 7 exercises


Exercise,Findings,Rules,Severity
r01/e00,0,--,--
r01/e01,0,--,--
r01/e02,1,DRAFT-001,MEDIUM
r01/e03,1,DRAFT-001,MEDIUM
r01/e04,1,DRAFT-001,MEDIUM
r01/e05,1,DRAFT-001,MEDIUM
r01/e06,2,DRAFT-001,MEDIUM
r01/e07,1,DRAFT-001,MEDIUM
r01/e08,1,DRAFT-001,MEDIUM


In [14]:
# Refined: only flag when the SAME key is read from different sources
@detect("DRAFT-002", severity="high")
def multi_source_same_key(g):
    """Flag endpoints where the same key is read from different sources."""
    for handler, route, accs in g.by_endpoint():
        key_sources: dict[str, set] = {}
        for a in accs:
            if a.key_literal:
                key_sources.setdefault(a.key_literal, set()).add(a.source)
        confused = {k: srcs for k, srcs in key_sources.items() if len(srcs) > 1}
        if confused:
            evidence = [a for a in accs if a.key_literal in confused]
            keys_str = ", ".join(sorted(confused))
            yield finding(
                f"Key confusion in {route.handler_name}: {keys_str}",
                evidence=evidence,
                endpoint=route,
            )

refined_results = batch_run(multi_source_same_key, graphs)
print(f"Refined rule: {refined_results.total} findings across {len(refined_results.hits)} exercises")
print(f"Reduction: {draft_results.total} -> {refined_results.total} findings")
refined_results

Refined rule: 7 findings across 6 exercises
Reduction: 8 -> 7 findings


Exercise,Findings,Rules,Severity
r01/e00,0,--,--
r01/e01,0,--,--
r01/e02,1,DRAFT-002,HIGH
r01/e03,0,--,--
r01/e04,1,DRAFT-002,HIGH
r01/e05,1,DRAFT-002,HIGH
r01/e06,2,DRAFT-002,HIGH
r01/e07,1,DRAFT-002,HIGH
r01/e08,1,DRAFT-002,HIGH


In [15]:
# Inspect the refined rule directly in the workbench
Explorer(g, run_fn(multi_source_same_key, g), target_path=t.r01.e03.path)

## Tracing Evidence

`finding_paths()` shows the call chain from an endpoint handler
to each piece of evidence.

In [16]:
r = scan(t.r01.e03)
for f in r.findings:
    print(f"\n{f.rule_id}: {f.title}")
    paths = finding_paths(f, g)
    for ev, path in zip(f.evidence, paths):
        label = getattr(ev, "raw_code", str(ev))[:60]
        print(f"  {label}")
        print(f"    path: {path}")


CONF-005: Dict merge with user-controlled data: {**user_data, **safe_order_data}
  {**user_data, **safe_order_data}
    path: ['routes.checkout_cart']

CONF-007: Conditional source selection with dict merge
  request.json
    path: ['routes.checkout_cart']
  request.form
    path: ['routes.checkout_cart']


## Evidence Source Code

Inspect the actual source lines for each piece of evidence.

In [17]:
# Show source context for all evidence in the first finding
f0 = r.findings[0]
print(f"Finding: {f0.title}\n")
for ev in f0.evidence:
    if hasattr(ev, "location"):
        show_source(ev)
        print()

Finding: Dict merge with user-controlled data: {**user_data, **safe_order_data}



## Registering a Rule

Pass `register=True` to add your rule to the global registry.
It will then run alongside built-in rules when you call `scan()`.

In [18]:
@detect("MY-003", severity="high", register=True)
def missing_key_validation(g):
    """Flag endpoints with direct source access but no key-level gets.

    If a handler passes request.form directly to a function without
    first validating individual keys, arbitrary fields can leak through.
    """
    for handler, route, accs in g.by_endpoint():
        direct = [a for a in accs if a.accessor == AccessorKind.DIRECT]
        keyed = [a for a in accs if a.key_literal is not None]
        if direct and not keyed:
            yield finding(
                f"No key validation in {route.handler_name}",
                evidence=direct,
                endpoint=route,
            )

# Now scan() includes MY-003 automatically
r = scan(t.r01.e03)
print("All findings (including MY-003):")
for f in r.findings:
    print(f"  {f.rule_id}: {f.title}")

All findings (including MY-003):
  CONF-005: Dict merge with user-controlled data: {**user_data, **safe_order_data}
  CONF-007: Conditional source selection with dict merge
  MY-003: No key validation in checkout_cart
